In [1]:
#import libraries
import pandas as pd
import json
import sys
sys.path.append("../src")
from genai_utils import get_groq_client

In [6]:
# Load result tables
incrementality_summary = pd.read_csv("../outputs/tables/incrementality_summary.csv")
incrementality_test = pd.read_csv(
    "../outputs/tables/incrementality_statistical_test.csv"
)
segment_incrementality = pd.read_csv(
    "../outputs/tables/segment_incrementality_summary.csv"
)
targeting_evaluation = pd.read_csv(
    "../outputs/tables/customer_targeting_model_evaluation.csv"
)

# Load merchant data + NLP theme summary
merchant_engagement = pd.read_csv("../data/samples/yelp_merchant_engagement.csv")
merchant_priority = pd.read_csv(
    "../data/samples/yelp_merchant_priority_final.csv"
)
merchant_themes = pd.read_csv(
    "../outputs/tables/merchant_theme_summary.csv"
)

# Criteo metrics
criteo_control_cr = float(
    incrementality_test.loc[
        incrementality_test["Metric"] == "Control Conversion Rate", "Value"
    ].values[0]
)
criteo_treatment_cr = float(
    incrementality_test.loc[
        incrementality_test["Metric"] == "Treatment Conversion Rate", "Value"
    ].values[0]
)
criteo_rel_lift = float(
    incrementality_test.loc[
        incrementality_test["Metric"] == "Relative Conversion Lift", "Value"
    ].values[0]
)
criteo_p_val = float(
    incrementality_test.loc[
        incrementality_test["Metric"] == "P-Value", "Value"
    ].values[0]
)

# Top segment
top_seg = segment_incrementality[
    segment_incrementality["attractive_segment"] == True
].iloc[0]

# Merchant engagement metrics (updated to 'priority_group')
tier_counts = merchant_priority["priority_group"].value_counts().to_dict()

# Top 3 CX drivers (aggregated across individual theme indicator columns)
theme_cols = [
    'cleanliness_ambiance',
    'food_product_quality',
    'order_accuracy_wait_time',
    'other_none',
    'pricing_value',
    'service_speed',
    'staff_behavior'
]

declining_themes = merchant_themes[
    merchant_themes["merchant_status"] == "Declining"
].copy()

theme_counts = declining_themes[theme_cols].sum().reset_index()
theme_counts.columns = ["theme", "merchant_count"]
total_theme_mentions = theme_counts["merchant_count"].sum()

theme_counts["pct_of_declining"] = (
    (theme_counts["merchant_count"] / total_theme_mentions * 100)
    .round(2).astype(str) + "%"
)

top_3_cx_drivers = (
    theme_counts.sort_values(by="merchant_count", ascending=False)
    .head(3)
    .to_dict(orient="records")
)

# Bundle into unified payload
executive_summary_data = {
    "criteo_targeting_campaign": {
        "control_conversion_rate": round(criteo_control_cr, 6),
        "treatment_conversion_rate": round(criteo_treatment_cr, 6),
        "relative_conversion_lift": f"{criteo_rel_lift * 100:.2f}%",
        "p_value": criteo_p_val,
        "is_statistically_significant": criteo_p_val < 0.05,
        "top_performing_segment": {
            "feature": str(top_seg["feature"]),
            "range": str(top_seg["segment"]),
            "treatment_conversion_rate": round(
                float(top_seg["treatment_conversion_rate"]), 4
            ),
            "control_conversion_rate": round(
                float(top_seg["control_conversion_rate"]), 4
            ),
            "relative_lift": f"{float(top_seg['relative_lift']) * 100:.2f}%",
        },
    },
    "yelp_merchant_engagement": {
        "priority_tiers": tier_counts,
        "top_declining_cx_drivers": top_3_cx_drivers,
    },
}

print("Executive summary data prepared successfully:")
print(json.dumps(executive_summary_data, indent=2))

Executive summary data prepared successfully:
{
  "criteo_targeting_campaign": {
    "control_conversion_rate": 0.001938,
    "treatment_conversion_rate": 0.003089,
    "relative_conversion_lift": "59.45%",
    "p_value": 7.308263285838679e-179,
    "is_statistically_significant": true,
    "top_performing_segment": {
      "feature": "f8",
      "range": "(3.634, 3.911]",
      "treatment_conversion_rate": 0.0115,
      "control_conversion_rate": 0.0075,
      "relative_lift": "54.23%"
    }
  },
  "yelp_merchant_engagement": {
    "priority_tiers": {
      "Expand": 147,
      "Growth Opportunity": 79,
      "Monitor / Intervene": 53,
      "Reassess": 21
    },
    "top_declining_cx_drivers": [
      {
        "theme": "food_product_quality",
        "merchant_count": 597,
        "pct_of_declining": "45.06%"
      },
      {
        "theme": "staff_behavior",
        "merchant_count": 317,
        "pct_of_declining": "23.92%"
      },
      {
        "theme": "pricing_value",
     

In [7]:
# Initialize Groq client
client = get_groq_client()

# Build prompt dynamically using executive_summary_data references
prompt = f"""
You are a Principal Data Scientist and Business Strategist writing an executive brief for C-suite leadership.

Synthesize the provided analysis results into a structured, highly actionable Executive Brief covering three core strategic pillars.

### Input Data
{json.dumps(executive_summary_data, indent=2)}

### Output Structure Requirements
Write a concise executive briefing with exactly three formal sections:

#### SECTION 1: Criteo Ad Incrementality & Precision Targeting Impact
* Detail baseline control vs. treatment conversion rates ({executive_summary_data['criteo_targeting_campaign']['control_conversion_rate']} vs {executive_summary_data['criteo_targeting_campaign']['treatment_conversion_rate']}), relative lift ({executive_summary_data['criteo_targeting_campaign']['relative_conversion_lift']}), and statistical confidence (p-val: {executive_summary_data['criteo_targeting_campaign']['p_value']}).
* Highlight the top-performing segment ({executive_summary_data['criteo_targeting_campaign']['top_performing_segment']['feature']} range {executive_summary_data['criteo_targeting_campaign']['top_performing_segment']['range']}) achieving {executive_summary_data['criteo_targeting_campaign']['top_performing_segment']['relative_lift']} relative lift.
* State strategic implications for ad spend reallocation toward high-intent cohorts.

#### SECTION 2: Yelp Merchant Risk & Operational CX Drivers
* Summarize merchant priority tier breakdowns ({executive_summary_data['yelp_merchant_engagement']['priority_tiers']}).
* Discuss top root-cause Customer Experience (CX) drivers behind merchant engagement drop-off:
  {json.dumps(executive_summary_data['yelp_merchant_engagement']['top_declining_cx_drivers'], indent=2)}
* Connect negative operational themes (e.g., quality, service friction, pricing perception) to platform retention and churn risk.

#### SECTION 3: Strategic Recommendations & Action Plan
* Provide 3-4 concrete strategic recommendations for cross-functional execution (Product, Operations, Marketing).
"""

response = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[{"role": "user", "content": prompt}],
    temperature=0.3,
)

raw_narrative = response.choices[0].message.content
print(raw_narrative)

# Save raw output to markdown file
with open("../outputs/narratives/executive_narrative_raw.md", "w", encoding="utf-8") as f:
    f.write(raw_narrative)

2026-09-02 21:19:01,308 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


**Executive Brief – Q3 2026**

---

### SECTION 1: Criteo Ad Incrementality & Precision Targeting Impact  
- **Baseline performance** – Control conversion rate = 0.001938; Treatment conversion rate = 0.003089.  
- **Incremental lift** – +59.45 % relative increase; statistical confidence = p‑value 7.3 × 10⁻¹⁷⁹ ( p < 0.001 , significant).  
- **High‑value cohort** – Feature **f8** in the range **(3.634, 3.911]** delivered a **54.23 %** relative lift (treatment = 1.15 %, control = 0.75 %).  
- **Strategic implication** – The experiment proves that precision‑targeted Criteo audiences generate > ½‑point absolute lift in conversion at negligible risk. Re‑allocating a portion of the brand‑wide media budget toward the identified high‑intent segment (f8) will maximize ROI while preserving overall spend efficiency.

---

### SECTION 2: Yelp Merchant Risk & Operational CX Drivers  
- **Merchant priority tier distribution**  
  - **Expand:** 147 merchants  
  - **Growth Opportunity:** 79 merchants

In [8]:
# Hand-polished executive narrative
cleaned_narrative = """
# Executive Briefing: Ad Incrementality & Merchant Retention Strategy

## 1. Criteo Ad Incrementality & Precision Targeting Impact

The Criteo randomized control trial (RCT) analysis demonstrates statistically significant ad incrementality. The treatment conversion rate was 0.31%, compared with 0.19% for the control group, representing a 59.45% relative conversion lift (p < 0.001).
Precision targeting analysis identified the `f8` segment in the range `(3.634, 3.911]` as a high-performing cohort, delivering a 54.23% relative lift. The corresponding treatment and control conversion rates were 1.15% and 0.75%, respectively.
These results support reallocating a portion of advertising spend toward high-performing, high-intent cohorts rather than relying exclusively on broad targeting. Future campaigns should prioritize segments demonstrating consistent incremental conversion performance while continuing to validate lift through controlled experimentation.

## 2. Yelp Merchant Risk & Operational CX Drivers

Merchant engagement analysis identified four priority groups:
- **Expand:** 147 merchants
- **Growth Opportunity:** 79 merchants
- **Monitor / Intervene:** 53 merchants
- **Reassess:** 21 merchants

Among Declining merchants, the leading Customer Experience (CX) themes identified through GenAI-assisted review analysis were:
- **food_product_quality:** 45.06% of total theme mentions
- **staff_behavior:** 23.92% of total theme mentions
- **pricing_value:** 10.42% of total theme mentions

These findings indicate that food and product quality is the dominant CX theme among Declining merchants, followed by staff behavior and pricing/value concerns. Because the extracted themes are based on review content, they provide qualitative context for the quantitative merchant engagement score and can help prioritize operational interventions.
Merchants in the **Monitor / Intervene** and **Reassess** groups should receive particular attention where declining engagement is accompanied by recurring negative CX themes. Addressing these issues can support merchant health, improve customer experience, and reduce potential platform retention risk.

## 3. Strategic Recommendations & Action Plan

1. **Prioritize high-incrementality advertising segments:** Reallocate testable portions of Criteo campaign spend toward high-performing cohorts such as the identified `f8` segment, while continuing controlled experiments to confirm incremental impact.
2. **Prioritize declining merchants using both health and CX signals:** Use the combination of `priority_group`, `merchant_status`, and extracted CX themes to identify merchants requiring intervention rather than relying on engagement score alone.
3. **Address food and service-related CX issues:** Focus merchant enablement and operational support on the dominant themes of `food_product_quality`, `staff_behavior`, and `pricing_value`, particularly among merchants classified as **Monitor / Intervene** or **Reassess**.
4. **Build a feedback loop between merchant health and CX analysis:** Incorporate recurring review themes into merchant monitoring workflows so that operational issues can be tracked alongside changes in engagement and sentiment over time.

## Bottom Line

The combined analysis provides two complementary growth levers: statistically validated advertising incrementality from Criteo and merchant-level engagement and CX diagnostics from Yelp. Precision targeting can improve the efficiency of customer acquisition, while theme-informed merchant interventions can help strengthen merchant health and retention.
"""

print(cleaned_narrative)

# Save final polished narrative
with open(
    "../outputs/narratives/executive_narrative.md",
    "w",
    encoding="utf-8"
) as f:
    f.write(cleaned_narrative)

print("Saved final narrative to ../outputs/narratives/executive_narrative.md")


# Executive Briefing: Ad Incrementality & Merchant Retention Strategy

## 1. Criteo Ad Incrementality & Precision Targeting Impact

The Criteo randomized control trial (RCT) analysis demonstrates statistically significant ad incrementality. The treatment conversion rate was 0.31%, compared with 0.19% for the control group, representing a 59.45% relative conversion lift (p < 0.001).
Precision targeting analysis identified the `f8` segment in the range `(3.634, 3.911]` as a high-performing cohort, delivering a 54.23% relative lift. The corresponding treatment and control conversion rates were 1.15% and 0.75%, respectively.
These results support reallocating a portion of advertising spend toward high-performing, high-intent cohorts rather than relying exclusively on broad targeting. Future campaigns should prioritize segments demonstrating consistent incremental conversion performance while continuing to validate lift through controlled experimentation.

## 2. Yelp Merchant Risk & Operat